In [1]:
import pandas as pd
import numpy as np

from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from imblearn.over_sampling import RandomOverSampler

In [2]:
seed = 10

In [ ]:
# Prepare train-test data
def prepare_train_test_data(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    bool_cols_train = X_train.select_dtypes(include=["bool"]).columns
    bool_cols_test = X_test.select_dtypes(include=["bool"]).columns

    if len(bool_cols_train) > 0:
        X_train[bool_cols_train] = X_train[bool_cols_train].astype(int)
    if len(bool_cols_test) > 0:
        X_test[bool_cols_test] = X_test[bool_cols_test].astype(int)

    X_train = pd.get_dummies(X_train, drop_first=False)
    X_test = pd.get_dummies(X_test, drop_first=False)

    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

    return X_train, X_test

In [ ]:
# Best parameters selected from cross-validation
best_xgb_params = {
    "mask_before": {
        "colsample_bytree": 0.6,
        "gamma": 0.1,
        "learning_rate": 0.05,
        "max_depth": 7,
        "min_child_weight": 1,
        "reg_lambda": 1.0,
        "subsample": 0.8
    },
    "mask_after": {
        "colsample_bytree": 0.6,
        "gamma": 1.0,
        "learning_rate": 0.05,
        "max_depth": 7,
        "min_child_weight": 5,
        "reg_lambda": 1.0,
        "subsample": 0.8
    },
    "general_before": {
        "colsample_bytree": 0.6,
        "gamma": 1.0,
        "learning_rate": 0.05,
        "max_depth": 7,
        "min_child_weight": 1,
        "reg_lambda": 1.0,
        "subsample": 0.8
    },
    "general_after": {
        "colsample_bytree": 0.6,
        "gamma": 1.0,
        "learning_rate": 0.05,
        "max_depth": 7,
        "min_child_weight": 5,
        "reg_lambda": 1.0,
        "subsample": 0.8
    }
}

best_rf_params = {
    "mask_before": {
        "criterion": "entropy",
        "max_depth": 20,
        "max_features": "sqrt",
        "min_samples_leaf": 1,
        "min_samples_split": 2
    },
    "mask_after": {
        "criterion": "entropy",
        "max_depth": 20,
        "max_features": "sqrt",
        "min_samples_leaf": 1,
        "min_samples_split": 2
    },
    "general_before": {
        "criterion": "entropy",
        "max_depth": 20,
        "max_features": "log2",
        "min_samples_leaf": 1,
        "min_samples_split": 2
    },
    "general_after": {
        "criterion": "entropy",
        "max_depth": 20,
        "max_features": "sqrt",
        "min_samples_leaf": 1,
        "min_samples_split": 2
    }
}

In [ ]:
# Load saved train-test datasets
def load_train_test(prefix):
    X_train = pd.read_csv(f"data/{prefix}_X_train.csv", keep_default_na=False)
    y_train = pd.read_csv(f"data/{prefix}_y_train.csv", keep_default_na=False).values.ravel()

    X_test = pd.read_csv(f"data/{prefix}_X_test.csv", keep_default_na=False)
    y_test = pd.read_csv(f"data/{prefix}_y_test.csv", keep_default_na=False).values.ravel()

    return X_train, y_train, X_test, y_test

In [ ]:
# Final validation: XGBoost
def final_validate_xgb(prefix, params, upsample=True):
    X_train, y_train, X_test, y_test = load_train_test(prefix)

    # Apply oversampling only to training data
    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_train, y_train = upsampler.fit_resample(X_train, y_train)

        if not isinstance(X_train, pd.DataFrame):
            X_train = pd.DataFrame(X_train)

    X_train_prepared, X_test_prepared = prepare_train_test_data(X_train, X_test)

    model = XGBClassifier(
        n_estimators=250,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=seed,
        tree_method="hist",
        **params
    )

    model.fit(X_train_prepared, y_train)

    y_pred = model.predict(X_test_prepared)
    y_prob = model.predict_proba(X_test_prepared)[:, 1]

    results = {
        "model": "XGBoost",
        "dataset": prefix,
        "auc": roc_auc_score(y_test, y_prob),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

    print(f"\nFinal validation results for XGBoost - {prefix}")
    print(results)
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=3))

    return model, results

In [ ]:
# Final validation: Random Forest
def final_validate_rf(prefix, params, upsample=True):
    X_train, y_train, X_test, y_test = load_train_test(prefix)

    # Apply oversampling only to training data
    if upsample:
        upsampler = RandomOverSampler(random_state=seed)
        X_train, y_train = upsampler.fit_resample(X_train, y_train)

        if not isinstance(X_train, pd.DataFrame):
            X_train = pd.DataFrame(X_train)

    X_train_prepared, X_test_prepared = prepare_train_test_data(X_train, X_test)

    model = RandomForestClassifier(
        n_estimators=250,
        bootstrap=True,
        random_state=seed,
        n_jobs=-1,
        **params
    )

    model.fit(X_train_prepared, y_train)

    y_pred = model.predict(X_test_prepared)
    y_prob = model.predict_proba(X_test_prepared)[:, 1]

    results = {
        "model": "Random Forest",
        "dataset": prefix,
        "auc": roc_auc_score(y_test, y_prob),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "accuracy": accuracy_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred)
    }

    print(f"\nFinal validation results for Random Forest - {prefix}")
    print(results)
    print("\nConfusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred, digits=3))

    return model, results

In [ ]:
# Run final validation for selected models
final_models = []
final_results = []

# XGBoost
for prefix in ["mask_before", "mask_after", "general_before", "general_after"]:
    model, results = final_validate_xgb(prefix, best_xgb_params[prefix], upsample=True)
    final_models.append((f"xgb_{prefix}", model))
    final_results.append(results)

# Random Forest
for prefix in ["mask_before", "mask_after", "general_before", "general_after"]:
    model, results = final_validate_rf(prefix, best_rf_params[prefix], upsample=True)
    final_models.append((f"rf_{prefix}", model))
    final_results.append(results)


Final validation results for XGBoost - mask_before
{'model': 'XGBoost', 'dataset': 'mask_before', 'auc': 0.8526846709672157, 'precision': 0.5818373812038015, 'recall': 0.7010178117048346, 'accuracy': 0.79188654353562, 'f1': 0.6358915175995383}

Confusion matrix:
[[1850  396]
 [ 235  551]]

Classification report:
              precision    recall  f1-score   support

           0      0.887     0.824     0.854      2246
           1      0.582     0.701     0.636       786

    accuracy                          0.792      3032
   macro avg      0.735     0.762     0.745      3032
weighted avg      0.808     0.792     0.798      3032


Final validation results for XGBoost - mask_after
{'model': 'XGBoost', 'dataset': 'mask_after', 'auc': 0.8918043561581205, 'precision': 0.8994252873563219, 'recall': 0.8765051806216746, 'accuracy': 0.8434283452098179, 'f1': 0.8878173308750532}

Confusion matrix:
[[1131  350]
 [ 441 3130]]

Classification report:
              precision    recall  f1-score

In [ ]:
# Save final validation results
final_validation_df = pd.DataFrame(final_results)
final_validation_df = final_validation_df.sort_values(
    by=["dataset", "auc"],
    ascending=[True, False]
).reset_index(drop=True)

final_validation_df

,model,dataset,auc,precision,recall,accuracy,f1
0,XGBoost,general_after,0.854730,0.887942,0.796721,0.779889,0.839862
1,Random Forest,general_after,0.842548,0.849722,0.875956,0.797902,0.862640
2,XGBoost,general_before,0.779480,0.724314,0.706285,0.701847,0.715186
3,Random Forest,general_before,0.777784,0.719018,0.729309,0.705475,0.724127
4,XGBoost,mask_after,0.891804,0.899425,0.876505,0.843428,0.887817
5,Random Forest,mask_after,0.885030,0.877221,0.912349,0.847783,0.894441
6,XGBoost,mask_before,0.852685,0.581837,0.701018,0.791887,0.635892
7,Random Forest,mask_before,0.851326,0.627876,0.590331,0.803100,0.608525


In [10]:
final_validation_df.to_csv("results/final_validation_results.csv", index=False)